# Step 0b: the SAMap homology maps from the phmmer hits

SAMap joins the two species through a table of homologous gene pairs with a score for each, in BLAST's tabular
layout. Here the pairs come from phmmer (HMMER 3.4, step 0a): every human protein searched against every mouse
protein and the other way round, hits kept at E <= 1e-5. This notebook names each hit's proteins by their gene
symbols (`gene_symbol:` in the Ensembl FASTA headers), keeps for each (query gene, target gene) pair the best bit
score over all their isoforms with that hit's E-value, and writes one line per pair. Columns other than the E-value
and bit score (identity, lengths, coordinates) are placeholders, as in the run of record.

This is `phmmer_to_samap_symbols.py` of the run of record (`record/0_homology/`): same parsing, same order of
chunks and lines, same output format, so the same hits give byte-identical maps (checked by `compare_with_record`).

Writes `OUT/0_homology/hsmm/hs_to_mm.txt` (human queries), `mm_to_hs.txt` (mouse queries) and `homology.json`
(counts, known orthologs found, checksums), first into `hsmm.partial/`, renamed when complete.

In [ ]:
# Parameters; papermill -p NAME VALUE overrides them in a cell below this one.
OUT = ""  # the run folder; this step writes OUT/0_homology/hsmm/
PHMMER_HITS = ""  # folder with hu_vs_mu/hu_<i>.tbl and mu_vs_hu/mu_<i>.tbl; "" means OUT/0_homology/phmmer (step 0a)
HUMAN_PROTEOME = "/n/lab_storage/eddy_lab/users/nhilgert/data/proteomes/fasta/Homo_sapiens.GRCh38.pep.all.fa"
MOUSE_PROTEOME = "/n/lab_storage/eddy_lab/users/nhilgert/data/proteomes/fasta/Mus_musculus.GRCm39.pep.all.fa"
N_CHUNKS = 32

In [ ]:
import datetime
import hashlib
import json
import os
import re
from pathlib import Path

In [ ]:
assert OUT, "set OUT, the run folder"
hits_dir = Path(PHMMER_HITS or Path(OUT) / "0_homology" / "phmmer")
out_dir = Path(OUT) / "0_homology" / "hsmm"
partial_dir = Path(OUT) / "0_homology" / "hsmm.partial"
assert not out_dir.exists(), f"{out_dir} exists; step 0b is done"
partial_dir.mkdir(parents=True, exist_ok=True)

# direction -> query and target proteomes, the hit tables' prefix, and the map SAMap reads
DIRECTIONS = {
    "hu_vs_mu": {"query": HUMAN_PROTEOME, "target": MOUSE_PROTEOME, "prefix": "hu", "map": "hs_to_mm.txt"},
    "mu_vs_hu": {"query": MOUSE_PROTEOME, "target": HUMAN_PROTEOME, "prefix": "mu", "map": "mm_to_hs.txt"},
}
GENE_SYMBOL = re.compile(r"gene_symbol:(\S+)")

# Orthologs a working map must hold (human, mouse), as the run of record's job checked.
KNOWN_ORTHOLOGS = [("HAND2", "Hand2"), ("FOXO1", "Foxo1"), ("IL11", "Il11"), ("WNT5A", "Wnt5a"), ("PGR", "Pgr")]


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


def protein_symbols(fasta):
    """Gene symbol of every protein with one in its FASTA header, and the number of proteins without."""
    symbols, without = {}, 0
    with open(fasta) as handle:
        for line in handle:
            if not line.startswith(">"):
                continue
            protein = line[1:].split(None, 1)[0]
            match = GENE_SYMBOL.search(line)
            if match:
                symbols[protein] = match.group(1)
            else:
                without += 1
    return symbols, without


def best_hits(table, query_symbols, target_symbols, best, counts):
    """Adds one phmmer --tblout file to best: (query gene, target gene) -> (bit score, E-value) of the best-scoring
    hit between their proteins. A later hit replaces an earlier one only with a strictly higher score, so ties keep
    the first, as in the run of record. tblout columns: target name, accession, query name, accession, E-value,
    score, ..."""
    with open(table) as handle:
        for line in handle:
            if line.startswith("#"):
                continue
            counts["lines"] += 1
            columns = line.split()
            if len(columns) < 6:
                continue
            target, query = columns[0], columns[2]
            evalue, score = float(columns[4]), float(columns[5])
            query_gene, target_gene = query_symbols.get(query), target_symbols.get(target)
            if query_gene is None:
                counts["dropped_query_without_symbol"] += 1
                continue
            if target_gene is None:
                counts["dropped_target_without_symbol"] += 1
                continue
            counts["kept"] += 1
            pair = (query_gene, target_gene)
            if pair not in best or score > best[pair][0]:
                best[pair] = (score, evalue)


def write_map(best, path):
    """One line per gene pair in BLAST's tabular layout (query, target, identity, length, mismatches, gap opens,
    query start, end, target start, end, E-value, bit score); only the last two carry information."""
    with open(path, "w") as handle:
        for (query_gene, target_gene), (score, evalue) in best.items():
            handle.write(f"{query_gene}\t{target_gene}\t100.0\t100\t0\t0\t1\t100\t1\t100\t{evalue}\t{score}\n")

In [ ]:
symbols = {path: protein_symbols(path) for path in (HUMAN_PROTEOME, MOUSE_PROTEOME)}
for path, (named, without) in symbols.items():
    print(f"{Path(path).name}: {len(named):,} proteins with a gene symbol, {without:,} without")

summary = {"written": datetime.datetime.now().isoformat(timespec="seconds"), "phmmer_hits": str(hits_dir),
           "proteomes": {Path(p).name: {"path": str(p), "sha256": sha256_of(p), "proteins_with_symbol": len(named),
                                        "proteins_without_symbol": without}
                         for p, (named, without) in symbols.items()}}

for direction, spec in DIRECTIONS.items():
    # Chunks in order 1..N_CHUNKS, so the map's lines come in the order of the run of record.
    tables = [hits_dir / direction / f"{spec['prefix']}_{i}.tbl" for i in range(1, N_CHUNKS + 1)]
    missing = [str(t) for t in tables if not t.is_file()]
    assert not missing, f"{len(missing)} hit tables missing, e.g. {missing[:2]}"

    best = {}
    counts = {"lines": 0, "kept": 0, "dropped_query_without_symbol": 0, "dropped_target_without_symbol": 0}
    for table in tables:
        best_hits(table, symbols[spec["query"]][0], symbols[spec["target"]][0], best, counts)
    write_map(best, partial_dir / spec["map"])

    pairs = set(best) if direction == "hu_vs_mu" else {(h, m) for (m, h) in best}
    found = {f"{h}/{m}": (h, m) in pairs for h, m in KNOWN_ORTHOLOGS}
    summary[direction] = {**counts, "gene_pairs": len(best), "known_orthologs_found": found,
                          "map": spec["map"], "map_sha256": sha256_of(partial_dir / spec["map"])}
    print(direction, json.dumps(summary[direction]))
    assert all(found.values()), f"{direction}: known orthologs missing from the map: {found}"

(partial_dir / "homology.json").write_text(json.dumps(summary, indent=1) + "\n")
os.replace(partial_dir, out_dir)
print(f"wrote {out_dir}: {sorted(p.name for p in out_dir.iterdir())}")